In [15]:
from sklearn.cluster import KMeans
import numpy as np
import pandas as pd
from sklearn.metrics import pairwise_distances

In [16]:
bin_path = '../data/movie_embeddings.bin'

with open(bin_path, 'rb') as f:
    shape = np.fromfile(f, dtype=np.uint32, count=2)
    num_movies, vector_dim = shape[0], shape[1]
    
    data = np.fromfile(f, dtype=np.float32)
    
    embeddings_matrix = data.reshape(num_movies, vector_dim)

In [17]:
kmeans = KMeans(n_clusters=15, init='k-means++', random_state=0)
kmeans.fit(embeddings_matrix)

KMeans(n_clusters=15, random_state=0)

In [18]:
distances = pairwise_distances(kmeans.cluster_centers_, embeddings_matrix, metric='cosine')

closest_indices = np.argmin(distances, axis=1)

df_mapping = pd.read_csv('../data/movie_mapping.csv')

starter_movies = df_mapping.iloc[closest_indices].copy()

display(starter_movies)
starter_ids = starter_movies['id'].tolist()
print("\nСписок ID для обновления базы данных:")
print(starter_ids)

,id,title
3193,359364,Человек
6930,13012,Преступник
7984,253450,Убийца
3568,356305,Почему он?
31,1368314,Пассажир
8512,533108,Очень скучная история
1056,49519,Семейка Крудс
13754,149937,Три истории
6483,9062,История любви
11730,11663,Обязательства



Список ID для обновления базы данных:
[359364, 13012, 253450, 356305, 1368314, 533108, 49519, 149937, 9062, 11663, 715931, 1116465, 6877, 228326, 1263249]


In [19]:
TOP_K_NEIGHBORS = 30 

sorted_indices = np.argsort(distances, axis=1)

starter_ids = []
starter_movies_list = []

df = pd.read_csv('../data/tmdb_movies_ru.csv')

for i in range(15):
    cluster_top_k_indices = sorted_indices[i, :TOP_K_NEIGHBORS]

    candidates = df.iloc[cluster_top_k_indices]

    best_candidate = candidates.sort_values(by='popularity', ascending=False).iloc[0]
    
    starter_ids.append(best_candidate['id'])
    starter_movies_list.append(best_candidate)

final_starters_df = pd.DataFrame(starter_movies_list)[['id', 'title', 'popularity']]
display(final_starters_df)

,id,title,popularity
2360,1477565,Полный провал: настоящий проект X,6.6840
1368,348893,Неоспоримый 4,9.3012
37,1235877,Народный герой,91.0798
1623,1261825,Боже. Как. Смешно.,8.4011
31,1368314,Пассажир,106.3277
1384,255709,Желание,9.2468
311,14160,Вверх,21.2211
456,337404,Круэлла,17.1396
965,537915,После,11.2567
304,1284016,Это хит!,21.3675
